In [ ]:
import json
import re
from pathlib import Path

import fitz
import pandas as pd

print('Libraries imported.')


In [ ]:
## 2. Define PDF File Lists and Update Rules

The later class-specific PDF files are authoritative and must override the older combined records for the same class and shift.


## 3. Extract Timetable Data from All PDFs

This rebuild reads all PDF files and writes the website-ready timetable bundle.


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location('build_dataset', 'build_dataset.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.extract_all_timetables()
print('Dataset rebuild finished.')


## 4. Validate Timetable Data for Completeness and Duplicates

Check the affected sections and ensure the class-specific update PDFs override the older combined entries without duplicates.


In [ ]:
with open('timetable_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

sections = data.get('sections', [])
required = [
    'Artificial Intelligence | BSAI-3A | Morning Shift',
    'Computer Science | BSCS-3A | Morning Shift',
    'Cybersecurity | BSCybSec-7A | Evening Shift',
    'Data Science | BSDS-3A | Morning Shift',
]

missing = [item for item in required if item not in sections]
assert not missing, f'Missing required sections: {missing}'
assert len(sections) == len(set(sections)), f'Duplicate section entries found: {sections}'

for item in required:
    print('Verified:', item)

print('Total unique sections:', len(sections))


In [ ]:
import json

path = 'timetable_data.json'
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

entries = []
for e in data['schedule_entries']:
    if e.get('department') == 'Cybersecurity' and e.get('section') == 'BSCybSec-7A':
        if e.get('shift') == 'Morning Shift' and e.get('file') == 'Updated BSCyber7th Evening.pdf':
            continue
        if e.get('shift') == 'Evening Shift' and e.get('file') == 'TT BSCyberSec M+E.pdf':
            continue
    entries.append(e)

data['schedule_entries'] = entries

data['sections'] = sorted({f"{e['department']} | {e['section']} | {e['shift']}" for e in entries})

catalog = []
for c in data.get('course_catalog', []):
    if c.get('department') == 'Cybersecurity' and c.get('section') == 'BSCybSec-7A':
        if c.get('shift') == 'Morning Shift' and c.get('file') == 'Updated BSCyber7th Evening.pdf':
            continue
        if c.get('shift') == 'Evening Shift' and c.get('file') == 'TT BSCyberSec M+E.pdf':
            continue
    catalog.append(c)

data['course_catalog'] = catalog

with open(path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print('BSCybSec-7A entries after cleanup:', len(entries))
print('BSCybSec-7A morning count:', sum(1 for e in entries if e.get('section') == 'BSCybSec-7A' and e.get('shift') == 'Morning Shift'))
print('BSCybSec-7A evening count:', sum(1 for e in entries if e.get('section') == 'BSCybSec-7A' and e.get('shift') == 'Evening Shift'))


In [ ]:
print('hello from notebook kernel')
